# Train and Tune FF MLP Model

In [1]:
from pathlib import Path

import optuna.visualization as vis
import pandas as pd

from config.config import Config
from src.data import time_series_split
from src.models.factory import Experiment
from src.models.mlp import MLP
from src.plots import plot_forecast_diagnostics, plot_forecast_overlay, plot_test_overlay, plot_val_overlay, plot_val_test_overlay
from src.runners import run_experiments
from src.utils import set_seed

In [2]:
cfg = Config(Path("../config/config.yaml"))
SEED = cfg.runtime.seed
HORIZON = cfg.runtime.horizon
rng = set_seed(SEED)

2025-08-28 17:42:02,135 - INFO - src.utils - Global random seed set to 42


In [3]:
df_full = pd.read_csv(Path(cfg.data.processed_dir) / "features_full.csv")

In [4]:
MODEL_NAME = "mlp"

experiments = [
    Experiment(
        name="mlp",
        build=lambda horizon, seed: MLP(horizon=horizon, random_state=seed),
        include_sentiment=True
    )
]

In [5]:
results = run_experiments(df_full, Path(cfg.data.processed_dir), experiments, HORIZON, SEED, n_trials=150, n_splits=5)

2025-08-28 17:42:02,235 - INFO - ModelTrainer - Initialized ModelTrainer for model: mlp
D:\IntelliJ\ml-stock-sent\.venv\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
D:\IntelliJ\ml-stock-sent\.venv\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2025-08-28 17:42:02,236] A new study created in memory with name: no-name-0fd73ebc-1758-4b70-99ce-7ba709975d92
2025-08-28 17:42:02,236 - INFO - ModelTrainer - Starting model tuning...
D:\IntelliJ\ml-stock-sent\.venv\Lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (128,) which is of type tuple.
  warnings.warn(message)
D:\IntelliJ\ml-stock-sent\.ve

In [ ]:
train, val, test, forecast = time_series_split(df_full, train_ratio=0.7, val_ratio=0.15, horizon=HORIZON)

In [ ]:
plot_test_overlay(test, results, Path(cfg.data.fig_dir) / f"{MODEL_NAME}_actual_vs_predicted_adj_close.png")
plot_val_overlay(val, results, Path(cfg.data.fig_dir) / f"val_{MODEL_NAME}_actual_vs_predicted_adj_close.png")
plot_val_test_overlay(val, test, results, Path(cfg.data.fig_dir) / f"val_test_{MODEL_NAME}_actual_vs_predicted_adj_close.png")
plot_forecast_overlay(test, forecast, results, Path(cfg.data.fig_dir) / f"{MODEL_NAME}_forecast.png")
plot_forecast_diagnostics(forecast, test, results, Path(cfg.data.fig_dir) / f"{MODEL_NAME}_forecast_diagnostics.png")

In [9]:
pd.DataFrame(results[0]["best_params"])

,hidden_layer_sizes,activation,solver,alpha,max_iter,tol,learning_rate_init,learning_rate,batch_size,early_stopping,n_iter_no_change,validation_fraction,shuffle,random_state
0,256,relu,adam,1.068326e-07,1000,0.000011,0.001499,adaptive,64,True,22,0.15,False,42
1,128,relu,adam,1.068326e-07,1000,0.000011,0.001499,adaptive,64,True,22,0.15,False,42
2,64,relu,adam,1.068326e-07,1000,0.000011,0.001499,adaptive,64,True,22,0.15,False,42


In [10]:
pd.DataFrame(results[0]["metrics"]["test"], index=[0])

,mae,mse,rmse,smape,r2
0,0.007674,0.000101,0.010063,1.675869,-0.018551


In [11]:
study = results[0]["study"]

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()
vis.plot_slice(study).show()
vis.plot_parallel_coordinate(study).show()
vis.plot_contour(study).show()
vis.plot_edf(study).show()

[W 2025-08-28 17:45:38,244] Param activation unique value length is less than 2.
[W 2025-08-28 17:45:38,244] Param activation unique value length is less than 2.
[W 2025-08-28 17:45:38,244] Param learning_rate unique value length is less than 2.
[W 2025-08-28 17:45:38,245] Param activation unique value length is less than 2.
[W 2025-08-28 17:45:38,245] Param activation unique value length is less than 2.
[W 2025-08-28 17:45:38,245] Param activation unique value length is less than 2.
[W 2025-08-28 17:45:38,246] Param activation unique value length is less than 2.
[W 2025-08-28 17:45:38,246] Param solver unique value length is less than 2.
[W 2025-08-28 17:45:38,246] Param activation unique value length is less than 2.
[W 2025-08-28 17:45:38,246] Param activation unique value length is less than 2.
[W 2025-08-28 17:45:38,247] Param learning_rate unique value length is less than 2.
[W 2025-08-28 17:45:38,247] Param solver unique value length is less than 2.
[W 2025-08-28 17:45:38,247] Pa